# Load data + import thư viện

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
def read_parquet_by_type(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
    purchase_history_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
    item_chunk_files = [file for file in files if 'item_chunk' in file]
    
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
    purchase_history_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_history_chunk_files]) if purchase_history_chunk_files else None
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
    
    # Trả về một dictionary chứa các DataFrame
    return {
        "user_chunk": user_chunk_df,
        "purchase_history_chunk": purchase_history_chunk_df,
        "item_chunk": item_chunk_df
    }

In [22]:
data = read_parquet_by_type("C:/Users/tncn2/Downloads/preprocessed-dataset")
df = data["item_chunk"]

print(df.shape)
print(df.columns)
df.head()

(27331, 14)
['item_id', 'price', 'category_l1', 'category', 'brand_final', 'target_user_group_final', 'item_type_final', 'color_final', 'origin_final', 'material_final', 'sale_status', 'description_merge', 'age_bucket_final', 'price_segment']


item_id,price,category_l1,category,brand_final,target_user_group_final,item_type_final,color_final,origin_final,material_final,sale_status,description_merge,age_bucket_final,price_segment
str,"decimal[38,4]",str,str,str,str,str,str,str,str,i32,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Sơ sinh""",null,"""Đen""","""Ý""","""Silicone""",0,"""Chi tiết sản phẩm …","""1-3M""","""Mid"""
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""",null,null,null,0,null,"""2-4Y""","""Mid"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Bé Trai""",null,"""Hồng""","""Đức""","""Silicone""",0,"""Chi tiết sản phẩm …",null,"""Low"""
"""0020010000094""",401000.0000,"""Tã""","""Merries_Sơ Sinh""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿Tã dán Merries size S 82 miế…","""3-6M""","""High"""
"""0020010000098""",401000.0000,"""Tã""","""Merries_Tã Quần""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿﻿Bỉm tã quần Merries size M …","""6-9M""","""High"""


# Bước 1
Gom nhóm giá các mặt hàng có category_l1 = 'tã' theo các nhãn "Bình dân", "Trung cấp" và "cao cấp".

In [23]:
# category_l1 = "Tã"
ta_df = df.filter(pl.col("category_l1") == "Tã")

ta_df = ta_df.with_columns(
    pl.col("price").cast(pl.Float64)
)
# Thống kê
stats = ta_df.select("price").describe()
print(stats)

shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ price         │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 442.0         │
│ null_count ┆ 0.0           │
│ mean       ┆ 301187.791855 │
│ std        ┆ 132196.214063 │
│ min        ┆ 1.0           │
│ 25%        ┆ 199000.0      │
│ 50%        ┆ 305000.0      │
│ 75%        ┆ 379000.0      │
│ max        ┆ 1.32e6        │
└────────────┴───────────────┘


In [24]:
import polars as pl
from sklearn.cluster import KMeans
import numpy as np

# Lọc các sản phẩm có category_l1 = "Tã"
ta_df = df.filter(pl.col("category_l1") == "Tã")

# Đảm bảo cột price là kiểu float
ta_df = ta_df.with_columns(pl.col("price").cast(pl.Float64))

# Gom cụm giá bằng K-Means (3 cụm)
X = np.array(ta_df["price"]).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X)

# Thêm cột "price_segment" chứa nhãn cụm
ta_df = ta_df.with_columns(pl.Series("price_segment", labels))

# Tính trung bình giá của từng cụm
cluster_mean = (
    ta_df.group_by("price_segment")
         .agg(pl.col("price").mean().alias("mean_price"))
         .sort("mean_price")
)

# Tạo mapping từ cluster → nhãn 1, 2, 3 (tăng dần theo giá trung bình)
mapping = {row["price_segment"]: i + 1 for i, row in enumerate(cluster_mean.iter_rows(named=True))}

# Gán lại nhãn 1, 2, 3 vào cột price_segment
price_segment_mapped = [mapping[c] for c in ta_df["price_segment"].to_list()]
ta_df = ta_df.with_columns(pl.Series("price_segment", price_segment_mapped))

# In kết quả
print("Kết quả gom cụm theo giá cho category_l1 = 'Tã':")
ta_df.select(["category_l1", "price", "price_segment"]).head(10)


Kết quả gom cụm theo giá cho category_l1 = 'Tã':


category_l1,price,price_segment
str,f64,i64
"""Tã""",401000.0,2
"""Tã""",401000.0,2
"""Tã""",355000.0,2
"""Tã""",379000.0,2
"""Tã""",401000.0,2
"""Tã""",319000.0,2
"""Tã""",401000.0,2
"""Tã""",401000.0,2
"""Tã""",89000.0,1


In [25]:
# Đếm số lượng sản phẩm thuộc từng nhóm price_segment
segment_counts = (
    ta_df
    .group_by("price_segment")
    .agg(pl.count().alias("so_san_pham"))
    .sort("price_segment")
)

print("\nTổng số sản phẩm theo từng nhóm giá:")
print(segment_counts)


Tổng số sản phẩm theo từng nhóm giá:
shape: (3, 2)
┌───────────────┬─────────────┐
│ price_segment ┆ so_san_pham │
│ ---           ┆ ---         │
│ i64           ┆ u32         │
╞═══════════════╪═════════════╡
│ 1             ┆ 158         │
│ 2             ┆ 282         │
│ 3             ┆ 2           │
└───────────────┴─────────────┘


C:\Users\tncn2\AppData\Local\Temp\ipykernel_2860\3845280442.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("so_san_pham"))


In [26]:
# Thống kê min, max, mean, median của từng cụm giá
check_stats = (
    ta_df
    .group_by("price_segment")
    .agg([
        pl.count().alias("so_mat_hang"),
        pl.col("price").min().alias("min_price"),
        pl.col("price").median().alias("median_price"),
        pl.col("price").mean().alias("mean_price"),
        pl.col("price").max().alias("max_price"),
    ])
    .sort("price_segment")
)

print("\nThống kê giá theo từng cụm K-Means:")
print(check_stats)



Thống kê giá theo từng cụm K-Means:
shape: (3, 6)
┌───────────────┬─────────────┬───────────┬──────────────┬───────────────┬───────────┐
│ price_segment ┆ so_mat_hang ┆ min_price ┆ median_price ┆ mean_price    ┆ max_price │
│ ---           ┆ ---         ┆ ---       ┆ ---          ┆ ---           ┆ ---       │
│ i64           ┆ u32         ┆ f64       ┆ f64          ┆ f64           ┆ f64       │
╞═══════════════╪═════════════╪═══════════╪══════════════╪═══════════════╪═══════════╡
│ 1             ┆ 158         ┆ 1.0       ┆ 185000.0     ┆ 167379.772152 ┆ 259000.0  │
│ 2             ┆ 282         ┆ 279000.0  ┆ 379000.0     ┆ 368932.624113 ┆ 638000.0  │
│ 3             ┆ 2           ┆ 1.32e6    ┆ 1.32e6       ┆ 1.32e6        ┆ 1.32e6    │
└───────────────┴─────────────┴───────────┴──────────────┴───────────────┴───────────┘


C:\Users\tncn2\AppData\Local\Temp\ipykernel_2860\1711682065.py:6: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("so_mat_hang"),


# Bước 2: 
Thống kê xem khách hàng đó mua tã thuộc nhóm nào nhiều nhất để gán nhãn cho khách hàng đó.

In [27]:
dfs = read_parquet_by_type("C:/Users/tncn2/Downloads/preprocessed-dataset")
history_df = dfs["purchase_history_chunk"]
history_df = history_df.select(["customer_id", "item_id", "quantity"])

print("Tổng số dòng:", history_df.shape[0])
history_df.head(10)

Tổng số dòng: 35729825


customer_id,item_id,quantity
i32,str,i32
5254214,"""7115000000004""",1
7573232,"""0029130000030""",1
8187418,"""3496000000053""",2
8187418,"""2700000000002""",2
6931560,"""0029110000036""",1
2262570,"""3953000000286""",1
3353278,"""2242000910001""",1
6674236,"""0007180000027""",1
5938241,"""1512000000004""",1


In [28]:
join_df = history_df.join(
    ta_df.select(["item_id", "price_segment"]),
    on="item_id",
    how="inner"
)

print("Sau khi join, số dòng:", join_df.shape[0])

Sau khi join, số dòng: 3169373


In [29]:
join_df.head(10)

customer_id,item_id,quantity,price_segment
i32,str,i32,i64
3353278,"""2242000910001""",1,2
4810993,"""2263000000021""",2,1
5049831,"""2242000910001""",1,2
7081375,"""6768000000005""",1,2
4927421,"""2265000000025""",1,2
6191567,"""2265000000022""",1,2
8184382,"""5836000000007""",2,1
6911174,"""2265000000024""",1,2
8187072,"""6766000000002""",1,1


In [30]:
# 2. Tính tổng số lượng mua theo customer_id và price_segment
agg_df = (
    join_df
    .group_by(["customer_id", "price_segment"])
    .agg(pl.col("quantity").sum().alias("total_quantity"))
)

In [31]:
# Bảng tổng hợp gốc
print("\nBảng tổng số lượng tã theo từng khách hàng và phân khúc:")
print(agg_df.sort("customer_id").head(30))

# 1. Top khách hàng mua nhiều nhất
print("\n=== Top 10 customer_id có tổng quantity cao nhất ===")
top10 = (
    agg_df.group_by("customer_id")
          .agg(pl.col("total_quantity").sum().alias("tong_quantity"))
          .sort("tong_quantity", descending=True)
          .head(10)
)
top10_join = agg_df.join(top10, on="customer_id", how="inner")
print(top10_join.sort(["tong_quantity", "customer_id"], descending=[True, False]).head(30))

# 2. 20 khách hàng ngẫu nhiên
print("\n=== In ra 10 customer_id ngẫu nhiên ===")
sample20 = agg_df.sample(n=10, seed=42)
print(sample20.sort("customer_id"))

# 3. Khách hàng có đủ 3 phân khúc (1, 2, 3)
print("\n=== Khách hàng có đủ 3 phân khúc tã khác nhau ===")
multi_seg = (
    agg_df.group_by("customer_id")
          .agg(pl.col("price_segment").n_unique().alias("so_phan_khuc"))
          .filter(pl.col("so_phan_khuc") == 3)
)
multi_seg_join = agg_df.join(multi_seg.select("customer_id"), on="customer_id", how="inner")
print(multi_seg_join.sort("customer_id").head(30))



Bảng tổng số lượng tã theo từng khách hàng và phân khúc:
shape: (30, 3)
┌─────────────┬───────────────┬────────────────┐
│ customer_id ┆ price_segment ┆ total_quantity │
│ ---         ┆ ---           ┆ ---            │
│ i32         ┆ i64           ┆ i32            │
╞═════════════╪═══════════════╪════════════════╡
│ 15126       ┆ 2             ┆ 7              │
│ 17309       ┆ 2             ┆ 1              │
│ 28879       ┆ 2             ┆ 6              │
│ 28986       ┆ 2             ┆ 2              │
│ 29041       ┆ 1             ┆ 1              │
│ …           ┆ …             ┆ …              │
│ 30776       ┆ 2             ┆ 7              │
│ 30910       ┆ 2             ┆ 10             │
│ 30910       ┆ 1             ┆ 2              │
│ 30963       ┆ 2             ┆ 2              │
│ 30988       ┆ 2             ┆ 8              │
└─────────────┴───────────────┴────────────────┘

=== Top 10 customer_id có tổng quantity cao nhất ===
shape: (19, 4)
┌─────────────┬──────────

In [35]:
# 3. Với mỗi customer_id, chọn phân khúc tã có quantity lớn nhất
#    => Đây là phân khúc "ưa thích" của khách hàng
ta_pref_df = (
    agg_df
    .sort(["customer_id", "total_quantity"], descending=[False, True])
    .group_by("customer_id")
    .agg(pl.col("price_segment").first().alias("diaper_segment_preference"))
)

In [36]:
print("Hoàn tất! Số khách hàng được gán nhãn:", ta_pref_df.shape[0])
ta_pref_df.head(10)

Hoàn tất! Số khách hàng được gán nhãn: 766978


customer_id,diaper_segment_preference
i32,i64
15126,2
17309,2
28879,2
28986,2
29041,2
29134,2
29346,2
29348,2
29390,2


In [38]:
# Thống kê số lượng khách hàng theo phân khúc tã ưa thích
print(ta_pref_df["diaper_segment_preference"].value_counts().sort("diaper_segment_preference"))


shape: (2, 2)
┌───────────────────────────┬────────┐
│ diaper_segment_preference ┆ count  │
│ ---                       ┆ ---    │
│ i64                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ 1                         ┆ 229430 │
│ 2                         ┆ 537548 │
└───────────────────────────┴────────┘
